In [1]:
import os
import tabula
import pandas as pd
import pytesseract
import cv2
import re
import numpy as np
from pathlib import Path
import pdf2image

In [ ]:

# Remove empty dataframes and those that are mostly NaN
cleaned_dataframes = []

for df in all_dataframes:
    # Skip if dataframe is empty
    if df.empty:
        continue
    
    # Skip if more than 50% of values are NaN
    if df.isna().sum().sum() / (len(df) * len(df.columns)) > 0.5:
        continue
    
    # Skip if all values in the first row are NaN
    if df.iloc[0].isna().all():
        continue
    
    cleaned_dataframes.append(df)

all_dataframes = cleaned_dataframes
print(f"Original: {len(all_dataframes)} dataframes | Cleaned: {len(cleaned_dataframes)} dataframes")

# Group dataframes by their columns and concatenate those with matching columns
grouped = {}

for df in all_dataframes:
    # Create a hashable key from the column names
    col_key = tuple(df.columns)
    
    if col_key not in grouped:
        grouped[col_key] = []
    grouped[col_key].append(df)

# Concatenate dataframes with matching columns
condensed_dataframes = []
for col_key, dfs in grouped.items():
    if len(dfs) > 0:
        concatenated_df = pd.concat(dfs, ignore_index=True)
        condensed_dataframes.append(concatenated_df)

all_dataframes = condensed_dataframes
print(f"Condensed from 187 dataframes to {len(all_dataframes)} dataframes")


Original: 7752 dataframes | Cleaned: 7752 dataframes


In [ ]:
# Create the project_csvs folder if it doesn't exist
base_path = r"C:\Users\nicho"
csv_folder = os.path.join(base_path, 'project_csvs')
os.makedirs(csv_folder, exist_ok=True)

# Save each dataframe as a CSV file
for i, df in enumerate(all_dataframes):
    print(all_dataframes[i].head())
    csv_filename = os.path.join(csv_folder, f'dataframe_{i}.csv')
    df.to_csv(csv_filename, index=False)
    

print(f"Saved {len(all_dataframes)} dataframes to {csv_folder}")

             0                                      source_file  page
0      REVISED  ALP-------------------------------__C30_5.0.pdf     1
1  10 FEB 7:30  ALP-------------------------------__C30_5.0.pdf     1
2      REVISED  ALP-------------------------------__C30_5.0.pdf     2
3  10 FEB 7:30  ALP-------------------------------__C30_5.0.pdf     2
4      REVISED  ALP-------------------------------__C30_5.0.pdf     3
                 0    1      2      3  \
0              NOC  Men  Women  Total   
1        AIN - AIN    1      2      3   
2    ALB - Albania    1      3      4   
3    AND - Andorra    2      3      5   
4  ARG - Argentina    1      2      3   

                                       source_file  page  
0  ALP-------------------------------__C30_5.0.pdf     1  
1  ALP-------------------------------__C30_5.0.pdf     1  
2  ALP-------------------------------__C30_5.0.pdf     1  
3  ALP-------------------------------__C30_5.0.pdf     1  
4  ALP-------------------------------_

In [119]:
pdf_path = r"C:/Users/nicho/pdfs/ALPWDH----------------------------__C25A_1.0.pdf"
pages = convert_from_path(pdf_path, dpi=400)

# Save first page as image
img = pages[0]
img.save("page.png")

In [120]:
# Load image
image = cv2.imread("page.png")

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Improve contrast
gray = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, 8
)

# OCR
raw_text = pytesseract.image_to_string(gray, config="--psm 6")
print(raw_text)


f> Alpine Skiin

We Sci ane | Ski ain ZO
[A Women's Downhill TER

MILANO CORTINEN Discesa donne / Descente femmes SS)
QLD

NN

FIS World Cup Standings
Classifica Coppa del Mondo FIS / Classement de la Coupe du monde FIS
As of FRI 30 JAN 2026
12 DEC 2025 St. Moritz (SUI)
13 DEC 2025 St. Moritz (SUI)
20 DEC 2025 Val d'lsére (FRA)
10 JAN 2026 Zauchensee (AUT)
Number of Competitors: 47, Number of NOCs: 12 17 JAN 2026 Tarvisio (ITA)
Code Code Points

1 537544 VONN Lindsey USA 100 80 60 100 60 400
2 507168 AICHER Emma GER 45 100 26 40 45 256 144
3 206668 WEIDLE-WINKELMANN Kira GER 32 18 80 22 80 232 168
4 299624 PIROVANO Laura ITA 32 40 45 50 40 207 193
5 299466 DELAGO Nicol ITA 22 - 29 29 100 180 220
5 298323 GOGGIA Sofia ITA 50 60 32 14 24 180 220
7 56128 HUETTER Cornelia AUT 40 15 100 11 13 179 221
8 6535455 JOHNSON Breezy USA 16 50 36 36 40 178 222
9 56125 PUCHNER Mirjam AUT 60 45 11 2 24 142 258
10 426257 LIE Kajsa Vickhoff NOR 12 13 18 80 18 141 259
11 565360 STUHEC Ilka SLO 18 22 50 1

In [121]:
def clean_token(t):
    """Clean OCR artifacts so tokens can be safely converted to int."""
    if t in ["-", "—", "–", "—", "·", "•"]:
        return None

    # Replace common OCR mistakes
    t = t.replace("O", "0")      # letter O → zero
    t = t.replace("o", "0")
    t = t.replace(",", "")       # remove commas
    t = t.replace(".", "")       # remove trailing periods
    t = t.replace("…", "")       # remove ellipsis
    t = t.strip()

    # If still not numeric, skip it
    if not t.isdigit():
        return None

    return int(t)

rows = []

for line in raw_text.splitlines():
    line = line.strip()
    if not line:
        continue

    # Extract Rank + FIS code
    m = re.match(r"^(\d+)\s+(\d+)\s+(.*)$", line)
    if not m:
        continue

    rank = int(m.group(1))
    fis = int(m.group(2))
    rest = m.group(3).strip()

    # Extract name + NOC
    m2 = re.match(r"(.+?)\s+([A-Z]{3})\s+(.*)$", rest)
    if not m2:
        continue

    name = m2.group(1).strip()
    noc = m2.group(2)
    tail = m2.group(3).strip()

    # Extract all numbers in the tail
    nums = re.findall(r"[0-9O\.]+", tail)
    nums = [clean_token(n) for n in nums if clean_token(n) is not None]

    if len(nums) == 0:
        continue

    wc_points = nums[-1]
    diff = nums[-2] if len(nums) > 1 else None

    # Remove last numbers from tail
    tail_clean = re.sub(r"[0-9O\.]+\s*$", "", tail)
    tail_clean = re.sub(r"[0-9O\.]+\s*$", "", tail_clean).strip()

    # Remaining tokens = event scores
    event_tokens = tail_clean.split()
    event_tokens = [clean_token(t) for t in event_tokens]

    # Normalize to exactly 5 events
    event_tokens = (event_tokens + [None]*5)[:5]

    rows.append([rank, fis, name, noc] + event_tokens + [wc_points, diff])

columns = [
    "Rank", "FIS_Code", "Name", "NOC",
    "Event1", "Event2", "Event3", "Event4", "Event5",
    "WC_Points", "Diff"
]

df = pd.DataFrame(rows, columns=columns)
df


,Rank,FIS_Code,Name,NOC,Event1,Event2,Event3,Event4,Event5,WC_Points,Diff
0,1,537544,VONN Lindsey,USA,100.0,80.0,60.0,100.0,NaN,400,60
1,2,507168,AICHER Emma,GER,45.0,100.0,26.0,40.0,45.0,144,256
2,3,206668,WEIDLE-WINKELMANN Kira,GER,32.0,18.0,80.0,22.0,80.0,168,232
3,4,299624,PIROVANO Laura,ITA,32.0,40.0,45.0,50.0,40.0,193,207
4,5,299466,DELAGO Nicol,ITA,22.0,NaN,29.0,29.0,100.0,220,180
5,5,298323,GOGGIA Sofia,ITA,50.0,60.0,32.0,14.0,24.0,220,180
6,7,56128,HUETTER Cornelia,AUT,40.0,15.0,100.0,11.0,13.0,221,179
7,8,6535455,JOHNSON Breezy,USA,16.0,50.0,36.0,36.0,40.0,222,178
8,9,56125,PUCHNER Mirjam,AUT,60.0,45.0,11.0,2.0,24.0,258,142
9,10,426257,LIE Kajsa Vickhoff,NOR,12.0,13.0,18.0,80.0,18.0,259,141


In [123]:
path = r"C:\Users\nicho\project_csvs\dataframe_11.csv"

df.to_csv(path, index=False)



In [ ]:

SEGMENT_LABEL = re.compile(r"([A-Za-z0-9\-]+):")

def parse_downhill_ocr(raw_text):
    athletes = []
    current = None

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue

        # Detect start of a new athlete row
        m = re.match(r"^(\d+)\s+(\d+)\s+(.+?)\s+([A-Z]{3})\s+(.*)$", line)
        if m:
            # Save previous athlete
            if current:
                athletes.append(current)

            rank = int(m.group(1))
            bib = int(m.group(2))
            name = m.group(3).strip()
            noc = m.group(4)
            tail = m.group(5)

            current = {
                "Rank": rank,
                "Bib": bib,
                "Name": name,
                "NOC": noc
            }

            # Parse the rest of the line for segments
            parse_segments_into(current, tail)
            continue

        # Continuation lines (same athlete)
        if current:
            parse_segments_into(current, line)

    # Append last athlete
    if current:
        athletes.append(current)

    return pd.DataFrame(athletes)


def parse_segments_into(row_dict, text):
    """
    Extracts segments like:
      Int1: 21.63 (13)
      I1-I2: 19.30 (1)
      Sp1: 90.33 (18)
      Fin: 1:36.10 (1)
    And stores them as:
      Int1_time, Int1_rank
      Sp1_speed, Sp1_rank
      Fin_time, Fin_rank
    """
    tokens = text.split()

    i = 0
    while i < len(tokens):
        # Detect segment label
        m = SEGMENT_LABEL.match(tokens[i])
        if m:
            seg = m.group(1)  # e.g., "Int1", "Sp2", "I3-I4"
            i += 1

            # Next token = time or speed
            if i < len(tokens):
                value = tokens[i]
                i += 1
            else:
                break

            # Next token may be "(rank)"
            rank = None
            if i < len(tokens) and re.match(r"\(\d+\)", tokens[i]):
                rank = int(tokens[i][1:-1])
                i += 1

            # Store cleanly
            row_dict[f"{seg}_value"] = value
            row_dict[f"{seg}_rank"] = rank

        else:
            i += 1


In [44]:
def run_ocr(image_path):
    image = cv2.imread(image_path)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    gray = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY,
        15, 8
    )

    return pytesseract.image_to_string(gray, config="--psm 6")



def parse_ocr(raw_text):
    rows = []

    def clean_num(tok):
        tok = tok.replace("O", "0").replace("o", "0")
        tok = tok.replace(",", "").replace(".", "").strip()
        if tok in ["-", "—", "–", "·", "•", ""]:
            return None
        return int(tok) if tok.isdigit() else None

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue

        # Find NOC (3 uppercase letters)
        m = re.search(r"\b([A-Z]{3})\b", line)
        if not m:
            continue

        noc = m.group(1)

        # Split into: left side (rank/fis/name), right side (numbers)
        left = line[:m.start()].strip()
        right = line[m.end():].strip()

        # Extract name = longest sequence of non-numeric tokens on left
        left_tokens = left.split()
        name_tokens = []
        for tok in left_tokens:
            if re.fullmatch(r"[0-9O]+", tok):
                continue
            name_tokens.append(tok)

        if not name_tokens:
            continue

        name = " ".join(name_tokens)

        # Extract all numbers on the right
        nums = [clean_num(t) for t in re.findall(r"[0-9O\.]+|[-—–]", right)]
        nums = [n for n in nums if n is not None]

        if len(nums) == 0:
            continue

        # WC points = last number
        wc_points = nums[-1]

        # Diff = second-to-last number (if present)
        diff = nums[-2] if len(nums) > 1 else None

        # Event scores = everything before last 1–2 numbers
        event_scores = nums[:-2] if diff is not None else nums[:-1]

        rows.append({
            "Name": name,
            "NOC": noc,
            "EventScores": event_scores,
            "WC_Points": wc_points,
            "Diff": diff
        })

    return pd.DataFrame(rows)

def process_pdf(pdf_path):
    # Skip Results Books entirely
    if "results_book" in pdf_path.lower():
        print(f"Skipping Results Book: {pdf_path}")
        return None

In [ ]:
def detect_results_table(ocr_text: str):
    """
    Detect whether a PDF page contains a Results table.
    Returns (bool, reason).
    """

    text = ocr_text.lower()

    # --- 1. Strong signals (almost always present on results pages) ---
    strong_keywords = [
        "results",          # page title
        "official results",
        "final results",
        "intermediate results",
        "did not start",    # DNS block
        "did not finish",   # DNF block
        "rank",             # table header
        "bib",              # table header
        "noc",              # nationality column
    ]

    for kw in strong_keywords:
        if kw in text:
            return True, f"Matched strong keyword: {kw}"

    # --- 2. Structural signals: table-like patterns ---
    # Rank column: 1, 2, 3, ... at line starts
    rank_pattern = r"^\s*\d+\s+\d+\s+[A-Z]{3}\s+"
    if re.search(rank_pattern, ocr_text, flags=re.MULTILINE):
        return True, "Matched rank/bib/NOC row pattern"

    # Time formats: 1:37.91, 0:58.22, etc.
    time_pattern = r"\b\d:\d{2}\.\d{2}\b"
    if re.search(time_pattern, text):
        return True, "Matched time format"

    # --- 3. Weak signals: athlete name + NOC ---
    # Example: "Breezy Johnson USA"
    noc_pattern = r"\b[A-Z]{3}\b"
    if len(re.findall(noc_pattern, ocr_text)) >= 5:
        return True, "Found multiple NOC codes"

    return False, "No results-table signals found"

def detect_page_type(text):
    t = text.lower()

    # Strong signals
    if "official results" in t or "final results" in t or "results" in t:
        return "C73"
    if "post event" in t or "analysis" in t:
        return "C77"

    # Fallback: detect results-like structure
    if re.search(r"\b[A-Z]{3}\b", t) and re.search(r"\d+:\d{2}\.\d{2}", t):
        return "RESULTS_GENERIC"

    return None


def scan_pdf_for_results(pdf_path):
    pages = pdf2image.convert_from_path(pdf_path, dpi=300)
    results = []

    for i, page in enumerate(pages):
        text = pytesseract.image_to_string(page)
        page_type = detect_page_type(text)

        if page_type:
            results.append({
                "page_number": i + 1,
                "page_type": page_type,
                "ocr_text": text
            })

    return results


def extract_table_from_ocr(text):
    lines = text.splitlines()

    # Identify header row (first row with 3+ uppercase tokens)
    header = None
    for line in lines:
        tokens = line.split()
        if sum(tok.isupper() for tok in tokens) >= 3:
            header = tokens
            break

    if not header:
        return None

    rows = []
    for line in lines:
        parts = line.split()
        if len(parts) >= len(header):
            rows.append(parts[:len(header)])

    df = pd.DataFrame(rows, columns=header)
    return df
def process_all_pdfs(pdf_dir):
    pdf_dir = Path(pdf_dir)
    output = {}

    for pdf in pdf_dir.glob("*.pdf"):
        print(f"\n📄 Scanning {pdf.name} ...")
        pages = scan_pdf_for_results(pdf)

        extracted = []
        for p in pages:
            df = extract_table_from_ocr(p["ocr_text"])
            if df is not None:
                extracted.append({
                    "page_number": p["page_number"],
                    "page_type": p["page_type"],
                    "table": df
                })
                print(f"  ➤ Extracted table on page {p['page_number']} ({p['page_type']})")

        output[pdf.name] = extracted

    return output



if __name__ == "__main__":
    base_path = r"C:\Users\nicho\pdfs"
    results = process_all_pdfs(base_path)

    print("\n\n🎉 DONE — Extracted tables from:")
    for pdf, pages in results.items():
        print(f"  {pdf}: {len(pages)} pages with tables")



📄 Scanning ALP-------------------------------__C08_1.0.pdf ...

📄 Scanning ALP-------------------------------__C30_5.0.pdf ...

📄 Scanning ALP-------------------------------__C93_11.0.pdf ...

📄 Scanning ALP-------------------------------__C95_11.0.pdf ...

📄 Scanning ALP-------------------------------__N02A_1.0.pdf ...
  ➤ Extracted table on page 3 (C73)

📄 Scanning ALPM------------------------------__C25B_2.0.pdf ...

📄 Scanning ALPM------------------------------__C26B_6.0.pdf ...

📄 Scanning ALPM------------------------------__C35_1.0.pdf ...

📄 Scanning ALPW------------------------------__C25B_2.0.pdf ...

📄 Scanning ALPW------------------------------__C26B_6.0.pdf ...

📄 Scanning ALPW------------------------------__C35_1.0.pdf ...

📄 Scanning ALPWDH----------------------------__C25A_1.0.pdf ...

📄 Scanning ALPWDH----------------------------__C26A_2.0.pdf ...

📄 Scanning ALPWDH----------------------------__C32C_4.0.pdf ...

📄 Scanning ALPWDH----------------------------__C77A_1.0.p

KeyboardInterrupt: 

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from pdfminer.high_level import extract_text
from multiprocessing import Pool, cpu_count
import joblib

def split_title_body(ocr_text):
    lines = [l.strip() for l in ocr_text.splitlines() if l.strip()]
    if not lines:
        return "", ""

    # Top 20% of lines = title region
    cutoff = max(1, int(len(lines) * 0.2))
    title = " ".join(lines[:cutoff])
    body = " ".join(lines[cutoff:])
    return title, body

def infer_label_from_filename(filename):
    m = re.search(r"__(C\d+[A-Z]?)_", filename)
    if m:
        return m.group(1)
    return "OTHER"

def build_training_data(pdf_dir):
    title_X, title_y = [], []
    body_X, body_y = [], []

    for pdf in Path(pdf_dir).glob("*.pdf"):
        label = infer_label_from_filename(pdf.name)
        print(f"Reading {pdf.name} → label {label}")

        pages = pdf2image.convert_from_path(pdf, dpi=200)

        for page in pages:
            text = pytesseract.image_to_string(page)
            title, body = split_title_body(text)

            title_X.append(title)
            title_y.append(label)

            body_X.append(body)
            body_y.append(label)

    return (title_X, title_y), (body_X, body_y)

def train_models(pdf_dir):
    (title_X, title_y), (body_X, body_y) = build_training_data(pdf_dir)

    title_model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=30000
        )),
        ("clf", LogisticRegression(max_iter=2000))
    ])

    body_model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=30000
        )),
        ("clf", LogisticRegression(max_iter=2000))
    ])

    print("\nTraining title classifier…")
    title_model.fit(title_X, title_y)

    print("Training body classifier…")
    body_model.fit(body_X, body_y)

    joblib.dump(title_model, "title_classifier.pkl")
    joblib.dump(body_model, "body_classifier.pkl")
    print("\n🎉 Saved title_classifier.pkl and body_classifier.pkl")
    return title_model, body_model
title_model, body_model = train_models(r"C:\Users\nicho\pdfs")

Reading ALP-------------------------------__C08_1.0.pdf → label C08
Reading ALP-------------------------------__C30_5.0.pdf → label C30
Reading ALP-------------------------------__C93_11.0.pdf → label C93
Reading ALP-------------------------------__C95_11.0.pdf → label C95
Reading ALP-------------------------------__N02A_1.0.pdf → label OTHER
Reading ALPM------------------------------__C25B_2.0.pdf → label C25B
Reading ALPM------------------------------__C26B_6.0.pdf → label C26B
Reading ALPM------------------------------__C35_1.0.pdf → label C35
Reading ALPW------------------------------__C25B_2.0.pdf → label C25B
Reading ALPW------------------------------__C26B_6.0.pdf → label C26B
Reading ALPW------------------------------__C35_1.0.pdf → label C35
Reading ALPWDH----------------------------__C25A_1.0.pdf → label C25A
Reading ALPWDH----------------------------__C26A_2.0.pdf → label C26A
Reading ALPWDH----------------------------__C32C_4.0.pdf → label C32C
Reading ALPWDH---------------

In [ ]:
import pdf2image
import pytesseract
import numpy as np
import cv2
import pandas as pd
from pathlib import Path
import joblib


def is_alpine_pdf(pdf_name):
    # Alpine skiing PDFs always start with ALP, ALPM, ALPW, ALPWDH, etc.
    return bool(re.match(r"^ALP", pdf_name))
    

# ---------------------------------------------------------
# 2. Split title/body from OCR text
# ---------------------------------------------------------
def split_title_body(ocr_text):
    lines = [l.strip() for l in ocr_text.splitlines() if l.strip()]
    if not lines:
        return "", ""
    cutoff = max(1, int(len(lines) * 0.2))
    return " ".join(lines[:cutoff]), " ".join(lines[cutoff:])


# ---------------------------------------------------------
# 3. ML page classifier (title‑weighted)
# ---------------------------------------------------------
def classify_page(ocr_text):
    title, body = split_title_body(ocr_text)

    title_probs = title_model.predict_proba([title])[0]
    body_probs  = body_model.predict_proba([body])[0]

    title_pred = title_model.classes_[title_probs.argmax()]
    body_pred  = body_model.classes_[body_probs.argmax()]

    if max(title_probs) > 0.65:
        return title_pred
    return body_pred


# ---------------------------------------------------------
# 4. Detect if a page visually contains a table
# ---------------------------------------------------------
def looks_like_table(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    lines = cv2.HoughLinesP(
        edges, 1, np.pi/180, threshold=120,
        minLineLength=200, maxLineGap=10
    )

    return lines is not None and len(lines) > 5


# ---------------------------------------------------------
# 5. OCR a single page
# ---------------------------------------------------------
def ocr_page(pdf_path, page_num, dpi=150):
    img = pdf2image.convert_from_path(
        pdf_path, dpi=dpi,
        first_page=page_num+1, last_page=page_num+1
    )[0]
    return img, pytesseract.image_to_string(img)


# ---------------------------------------------------------
# 6. Deduplicate DataFrame columns
# ---------------------------------------------------------
def dedupe_columns(df):
    seen = {}
    new_cols = []
    for col in df.columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    df.columns = new_cols
    return df


# ---------------------------------------------------------
# 7. Universal OCR table extractor
# ---------------------------------------------------------
def extract_table_from_ocr(text):
    lines = text.splitlines()

    header = None
    for line in lines:
        tokens = line.split()
        if sum(tok.isupper() for tok in tokens) >= 3:
            header = tokens
            break

    if not header:
        return None

    rows = []
    for line in lines:
        parts = line.split()
        if len(parts) >= len(header):
            rows.append(parts[:len(header)])

    if not rows:
        return None

    df = pd.DataFrame(rows, columns=header)
    return dedupe_columns(df)


# ---------------------------------------------------------
# 8. FAST PDF scanner (no full rendering)
# ---------------------------------------------------------
def scan_pdf_fast(pdf_path):
    info = pdf2image.pdfinfo_from_path(pdf_path)
    num_pages = info["Pages"]

    extracted = []

    for page_num in range(num_pages):
        # Load low‑DPI preview (fast)
        preview = pdf2image.convert_from_path(
            pdf_path, dpi=60,
            first_page=page_num+1, last_page=page_num+1
        )[0]

        # Skip pages without table structure
        if not looks_like_table(preview):
            continue

        # OCR only table‑like pages
        img, text = ocr_page(pdf_path, page_num)

        label = classify_page(text)
        if not label.startswith("C73"):  # C73, C73A, C73B
            continue

        df = extract_table_from_ocr(text)
        if df is not None:
            df["source_pdf"] = Path(pdf_path).name
            df["page"] = page_num + 1
            extracted.append(df)

    return extracted


# ---------------------------------------------------------
# 9. Process all PDFs in directory
# ---------------------------------------------------------
def process_all_pdfs(pdf_dir):
    pdf_dir = Path(pdf_dir)
    all_tables = []

    for pdf in pdf_dir.glob("*.pdf"):
        print(f"\nScanning {pdf.name}...")
        tables = scan_pdf_fast(pdf)
        all_tables.extend(tables)

    if not all_tables:
        return pd.DataFrame()

    return pd.concat(all_tables, ignore_index=True)


# ---------------------------------------------------------
# 10. RUN PIPELINE
# ---------------------------------------------------------
master_df = process_all_pdfs(r"C:\Users\nicho\pdfs")
print(master_df.head())
print(master_df.shape)



Scanning ALP-------------------------------__C08_1.0.pdf...

Scanning ALP-------------------------------__C30_5.0.pdf...

Scanning ALP-------------------------------__C93_11.0.pdf...

Scanning ALP-------------------------------__C95_11.0.pdf...

Scanning ALP-------------------------------__N02A_1.0.pdf...

Scanning ALPM------------------------------__C25B_2.0.pdf...

Scanning ALPM------------------------------__C26B_6.0.pdf...

Scanning ALPM------------------------------__C35_1.0.pdf...

Scanning ALPW------------------------------__C25B_2.0.pdf...

Scanning ALPW------------------------------__C26B_6.0.pdf...

Scanning ALPW------------------------------__C35_1.0.pdf...

Scanning ALPWDH----------------------------__C25A_1.0.pdf...

Scanning ALPWDH----------------------------__C26A_2.0.pdf...

Scanning ALPWDH----------------------------__C32C_4.0.pdf...

Scanning ALPWDH----------------------------__C77A_1.0.pdf...

Scanning ALPWDH----------------------------__C77B_1.0.pdf...

Scanning AL

In [48]:
import pdf2image
import pytesseract
import numpy as np
import cv2
import pandas as pd
from pathlib import Path
import joblib
import re


# ---------------------------------------------------------
# 1. Load models
# ---------------------------------------------------------
title_model = joblib.load(r"C:\Users\nicho\title_classifier.pkl")
body_model  = joblib.load(r"C:\Users\nicho\body_classifier.pkl")

# Only these page types contain real results tables
VALID_RESULTS = {"C73", "C73A", "C73B"}

def detect_sport(pdf_name):
    prefix = pdf_name.split("-")[0]  # everything before first dash
    if prefix.startswith("ALP"):
        return "alpine"
    if prefix.startswith("BOB"):
        return "bobsleigh"
    if prefix.startswith("BTH"):
        return "biathlon"
    if prefix.startswith("NCB"):
        return "nordic_combined"
    if prefix.startswith("CCS"):
        return "cross_country"
    if prefix.startswith("FSK"):
        return "figure_skating"
    if prefix.startswith("FRS"):
        return "moguls"
    if prefix.startswith("SKN"):
        return "skeleton"
    if prefix.startswith("LUG"):
        return "luge"
    if prefix.startswith("SBD"):
        return "snowboard"
    if prefix.startswith("SJP"):
        return "ski_jumping"
    if prefix.startswith("SSK"):
        return "speed_skating"
    if prefix.startswith("SMT"):
        return "ski_mountaineering"
    if prefix.startswith("STK"):
        return "short_track_speed_skating"
    return "unknown"

def extract_table_by_sport(sport, text):
    if sport == "alpine":
        return extract_alpine_table(text)
    if sport == "bobsleigh":
        return extract_bobsleigh_table(text)
    if sport == "speed_skating":
        return extract_speed_skating_table(text)
    if sport == "nordic_combined":
        return extract_nordic_combined_table(text)
    if sport == "cross_country":
        return extract_cross_country_table(text)
    if sport == "figure_skating":
        return extract_figure_skating_table(text)
    if sport == "moguls":
        return extract_moguls_table(text)
    if sport == "ski_jumping":
        return extract_ski_jumping_table(text)
    if sport == "snowboard":
        return extract_snowboard_table(text)
    if sport == "luge":
        return extract_luge_table(text)
    if sport == "skeleton":
        return extract_skeleton_table(text)
    if sport == "ski_mountaineering":
        return extract_ski_mountaineering_table(text)
    if sport == "biathlon":
        return extract_biathlon_table(text)
    if sport == "short_track_speed_skating":
        return extract_short_track_table(text)
    return None

def extract_alpine_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 8 and parts[0].isdigit() and ":" in " ".join(parts):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_bobsleigh_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 6:
            continue
        if parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[1]):
            rows.append(parts)
    if not rows:
        return None
    return pd.DataFrame(rows)

def extract_figure_skating_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 4 and parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[-2]):
            rows.append(parts)
    if not rows:
        return None
    return pd.DataFrame(rows)

def extract_nordic_combined_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 6 and parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[2]):
            rows.append(parts)
    if not rows:
        return None
    return pd.DataFrame(rows)

def extract_cross_country_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit() and ":" in " ".join(parts):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_speed_skating_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 4 and parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[-2]):
            rows.append(parts)
    if not rows:
        return None
    return pd.DataFrame(rows)

def extract_figure_skating_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 4 and parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[-2]):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_ski_jumping_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 6 and parts[0].isdigit() and "m" in " ".join(parts):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_moguls_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 6 and parts[0].isdigit() and parts[1].isdigit():
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None


def extract_snowboard_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit():
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_luge_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[1]):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_skeleton_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[1]):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_ski_mountaineering_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit() and ":" in " ".join(parts):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_biathlon_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 6 and parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[2]):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

def extract_short_track_table(text):
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit() and re.fullmatch(r"[A-Z]{3}", parts[1]):
            rows.append(parts)
    return pd.DataFrame(rows) if rows else None

# ---------------------------------------------------------
# 2. Split title/body
# ---------------------------------------------------------
def split_title_body(ocr_text):
    lines = [l.strip() for l in ocr_text.splitlines() if l.strip()]
    if not lines:
        return "", ""
    cutoff = max(1, int(len(lines) * 0.2))
    return " ".join(lines[:cutoff]), " ".join(lines[cutoff:])


# ---------------------------------------------------------
# 3. Strong classifier: only allow C73/C73A/C73B
# ---------------------------------------------------------
def classify_page(ocr_text):
    title, body = split_title_body(ocr_text)

    title_probs = title_model.predict_proba([title])[0]
    body_probs  = body_model.predict_proba([body])[0]

    title_pred = title_model.classes_[title_probs.argmax()]
    body_pred  = body_model.classes_[body_probs.argmax()]

    # Trust title if confident
    pred = title_pred if max(title_probs) > 0.65 else body_pred

    # Only allow valid results pages
    if pred not in VALID_RESULTS:
        return "OTHER"

    return pred


# ---------------------------------------------------------
# 4. Strict table detector: must have grid-like structure
# ---------------------------------------------------------
def looks_like_table(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 40, 120)

    lines = cv2.HoughLinesP(
        edges, 1, np.pi/180, threshold=100,
        minLineLength=150, maxLineGap=15
    )

    # Require at least 4 horizontal/vertical lines
    return lines is not None and len(lines) >= 4


# ---------------------------------------------------------
# 5. OCR a single page
# ---------------------------------------------------------
def ocr_page(pdf_path, page_num, dpi=150):
    img = pdf2image.convert_from_path(
        pdf_path, dpi=dpi,
        first_page=page_num+1, last_page=page_num+1
    )[0]
    return img, pytesseract.image_to_string(img)


# ---------------------------------------------------------
# 6. Deduplicate columns
# ---------------------------------------------------------
def dedupe_columns(df):
    seen = {}
    new_cols = []
    for col in df.columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
    df.columns = new_cols
    return df




# ---------------------------------------------------------
# 8. FAST PDF scanner
# ---------------------------------------------------------
def scan_pdf_fast(pdf_path):
    info = pdf2image.pdfinfo_from_path(pdf_path)
    pdf_name = Path(pdf_path).name 
    num_pages = info["Pages"]

    extracted = []

    for page_num in range(num_pages):
        # Low-DPI preview
        preview = pdf2image.convert_from_path(
            pdf_path, dpi=60,
            first_page=page_num+1, last_page=page_num+1
        )[0]

        if not looks_like_table(preview):
            continue

        img, text = ocr_page(pdf_path, page_num)

        label = classify_page(text)
        if label == "OTHER":
            continue

        sport = detect_sport(pdf_name)
        df = extract_table_by_sport(sport, text)
        if df is not None:
            df["source_pdf"] = Path(pdf_path).name
            df["page"] = page_num + 1
            extracted.append(df)

    return extracted


# ---------------------------------------------------------
# 9. Process all PDFs
# ---------------------------------------------------------
def process_all_pdfs(pdf_dir):
    pdf_dir = Path(pdf_dir)
    all_tables = []

    for pdf in pdf_dir.glob("*.pdf"):
        pdf_name = pdf.name

        print(f"\nScanning {pdf_name}...")
        tables = scan_pdf_fast(pdf)
        all_tables.extend(tables)

    if not all_tables:
        return pd.DataFrame()

    return pd.concat(all_tables, ignore_index=True)




# ---------------------------------------------------------
# 10. RUN PIPELINE
# ---------------------------------------------------------
master_df = process_all_pdfs(r"C:\Users\nicho\pdfs")
print(master_df.head())
print(master_df.shape)

mdf = master_df.dropna().reset_index(drop=True)
print(mdf.shape())


Scanning ALP-------------------------------__C08_1.0.pdf...

Scanning ALP-------------------------------__C30_5.0.pdf...

Scanning ALP-------------------------------__C93_11.0.pdf...

Scanning ALP-------------------------------__C95_11.0.pdf...

Scanning ALP-------------------------------__N02A_1.0.pdf...

Scanning ALPM------------------------------__C25B_2.0.pdf...

Scanning ALPM------------------------------__C26B_6.0.pdf...

Scanning ALPM------------------------------__C35_1.0.pdf...

Scanning ALPW------------------------------__C25B_2.0.pdf...

Scanning ALPW------------------------------__C26B_6.0.pdf...

Scanning ALPW------------------------------__C35_1.0.pdf...

Scanning ALPWDH----------------------------__C25A_1.0.pdf...

Scanning ALPWDH----------------------------__C26A_2.0.pdf...

Scanning ALPWDH----------------------------__C32C_4.0.pdf...

Scanning ALPWDH----------------------------__C77A_1.0.pdf...

Scanning ALPWDH----------------------------__C77B_1.0.pdf...

Scanning AL

TypeError: 'tuple' object is not callable

In [67]:

mask = master_df["source_pdf"].str.contains(r"__C73[A-B]?_", regex=True, na=False)
master_df_1 = master_df[mask].copy()

print(master_df_1.shape)
master_df.to_csv(r"C:\Users\nicho\extracted_tables.csv", index=False)
master_df_1.to_csv(r"C:\Users\nicho\results_tables_c73.csv", index=False)

def clean_cell(x):
    if pd.isna(x):
        return x
    
    x = str(x)

    # remove OCR junk words
    x = re.sub(r"\bedit\b", "", x)

    # fix weird symbols
    x = x.replace("«", "")
    x = x.replace("»", "")
    x = x.replace("©", "")
    x = x.replace("—", "-")
    
    # collapse whitespace
    x = re.sub(r"\s+", " ", x).strip()

    return x

cleaned_master_df = master_df.map(clean_cell)
cleaned_master_df

(65, 26)


,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,57,512527,MIGGIANO,Alessio,sul,105,1280,,95,(14),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,57,6203445,ALLIOD,Benjamin,Jacques,ITA,105,1280,82,(18),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,59,92720,POPOV,Albert,BUL,103,1282,:,103,(20),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,60,511996,YULE,Daniel,sul,102,1283,:,102,(21),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,61,202762,JOCHER,Simon,GER,1001285,=,38,(29),:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428,22,ISR,FIRESTONE,Jared,4.74,(22),45.78,(24),93.89,(19),...,58.15,(23),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
429,23,JPN,TAKAHASHI,Hiroatsu,4.70,(16),46.11,(18),93.60,(22),...,(21),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
430,24,CAN,BRUSIC,Josip,4.72(=18),46.01,(=20),93.11,(23),112.00,...,(22),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
431,12,FEB,2026,Milano,(ITA),NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [69]:
def classify_row(row):
    text = " ".join([str(x) for x in row if pd.notna(x)])

    # metadata rows
    if "FEB" in text or "2026" in text:
        return "meta"

    # performance split rows (many decimals)
    if len(re.findall(r"\d+\.\d+", text)) >= 3:
        return "splits"

    # athlete rows (has uppercase surname + country code)
    if re.search(r"\b[A-Z]{3}\b", text):
        return "athlete"

    return "unknown"
cleaned_master_df["row_type"] = cleaned_master_df.apply(classify_row, axis=1)
print(cleaned_master_df["row_type"].value_counts())


row_type
athlete    215
splits     157
unknown     57
meta         4
Name: count, dtype: int64


In [73]:
athletes = cleaned_master_df[cleaned_master_df["row_type"] == "athlete"]
athletes

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,row_type
1,57,6203445,ALLIOD,Benjamin,Jacques,ITA,105,1280,82,(18),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,athlete
2,59,92720,POPOV,Albert,BUL,103,1282,:,103,(20),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,athlete
4,61,202762,JOCHER,Simon,GER,1001285,=,38,(29),:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,athlete
5,62,104531,CRAWFORD,James,CAN,97,1288,=,25,(38),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,athlete
6,63,194935.,FAVROT,Thibaut,FRA,95,1290,-,:,95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,athlete
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392,48,NIKAIDO,Ren,JPN,82,ons,i,or,m7,war,...,oon,i,098,704,NaN,NaN,NaN,NaN,NaN,athlete
396,38,|,TOMASIAK,Kacper,POL,“ae,88,8,1,ms,...,DNS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,athlete
397,43,HOFEMANN,Feix,GER,ms,ors,73,088,ora,861,...,ims,16,“a8,4,NaN,NaN,NaN,NaN,NaN,athlete
398,45,|RAIMUND,Philipp,GER,a7,ono,73,08,758,v4,...,‘080,16,Orr,4,NaN,NaN,NaN,NaN,NaN,athlete


In [74]:
splits = cleaned_master_df[cleaned_master_df["row_type"] == "splits"]
splits

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,row_type
255,21,GRENIER,Valerie,CAN,Intt:,21.75,20.23,(23),031,Spt:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits
256,22,MUZAFERIJA,Elvedina,BIH,Intt:,21.76,19.95,(17),032,Spt:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits
257,23,GAUCHE,Laura,FRA,Intt:,21.69,20.13,(20),028,Spt:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits
258,24,WRIGHT,Isabella,USA,Intt:,21.63,20.20,(22),0-19,Spt:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits
259,25,CERUTTI,Camille,FRA,Intt:,22.61,20.87,(28),117,Spt:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426,20,USA,BAREFOOT,Daniel,4.72(=18),46.23,(16),94.14,(18),113.92,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits
427,21,AUS,TIMMINGS,Nicholas,4.75,(23),46.15,(17),91.83,(24),...,(24),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits
428,22,ISR,FIRESTONE,Jared,4.74,(22),45.78,(24),93.89,(19),...,(23),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits
429,23,JPN,TAKAHASHI,Hiroatsu,4.70,(16),46.11,(18),93.60,(22),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,splits


In [75]:
meta = cleaned_master_df[cleaned_master_df["row_type"] == "meta"]
meta

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,row_type
89,95,192746,THEAUX,Adrien,FRA,61,2026,,52,(34),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,meta
389,8,FEB,2026,/Livigno,Snow,Park,(ITA),6185,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,meta
431,12,FEB,2026,Milano,(ITA),NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,meta
432,12,FEB,2026,Milano,(ITA),NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,meta


In [ ]:
cleaned_master_df = cleaned_master_df[cleaned_master_df[23].isna()]
cleaned_master_df = cleaned_master_df[cleaned_master_df[17].isna()]




KeyError: 23

In [95]:
cleaned_master_df = cleaned_master_df.dropna(axis=1, how="all")
cleaned_master_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,source_pdf,page,14,15,16,row_type
0,57,512527,MIGGIANO,Alessio,sul,105,1280,,95,(14),:,10,(48),NaN,ALPM------------------------------__C25B_2.0.pdf,2,NaN,NaN,NaN,unknown
1,57,6203445,ALLIOD,Benjamin,Jacques,ITA,105,1280,82,(18),--,:,23,(37),ALPM------------------------------__C25B_2.0.pdf,2,NaN,NaN,NaN,athlete
2,59,92720,POPOV,Albert,BUL,103,1282,:,103,(20),:,NaN,NaN,NaN,ALPM------------------------------__C25B_2.0.pdf,2,NaN,NaN,NaN,athlete
3,60,511996,YULE,Daniel,sul,102,1283,:,102,(21),=,:,NaN,NaN,ALPM------------------------------__C25B_2.0.pdf,2,NaN,NaN,NaN,unknown
4,61,202762,JOCHER,Simon,GER,1001285,=,38,(29),:,62,(22),NaN,NaN,ALPM------------------------------__C25B_2.0.pdf,2,NaN,NaN,NaN,athlete
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428,22,ISR,FIRESTONE,Jared,4.74,(22),45.78,(24),93.89,(19),114.21,(19),119.44,(23),SKNMSINGLES-----------FNL---------__C77D_4.0.pdf,3,58.15,(23),NaN,splits
429,23,JPN,TAKAHASHI,Hiroatsu,4.70,(16),46.11,(18),93.60,(22),113.49(=21),119.68,(22),58.06,SKNMSINGLES-----------FNL---------__C77D_4.0.pdf,3,(21),NaN,NaN,splits
430,24,CAN,BRUSIC,Josip,4.72(=18),46.01,(=20),93.11,(23),112.00,(24),119.12,(24),58.14,SKNMSINGLES-----------FNL---------__C77D_4.0.pdf,3,(22),NaN,NaN,splits
431,12,FEB,2026,Milano,(ITA),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STKW500M--------------FNL---------__C73A_1.0.pdf,1,NaN,NaN,NaN,meta


In [96]:
cleaned_master_df.to_csv(r"C:\Users\nicho\cleaned_extracted_tables.csv", index=False)

(423, 20)

,0,1,2,3,4,5,6,7,8,9,...,199,200,201,202,203,204,205,206,207,row_text
0,57,512527,MIGGIANO,Alessio,sul,105,1280,,95,(14),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57 512527 MIGGIANO Alessio sul 105 1280 95 (1...
1,57,6203445,ALLIOD,Benjamin,Jacques,ITA,105,1280,82,(18),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57 6203445 ALLIOD Benjamin Jacques ITA 105 128...
2,59,92720,POPOV,Albert,BUL,103,1282,:,103,(20),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59 92720 POPOV Albert BUL 103 1282 : 103 (20) ...
3,60,511996,YULE,Daniel,sul,102,1283,:,102,(21),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60 511996 YULE Daniel sul 102 1283 : 102 (21) ...
4,61,202762,JOCHER,Simon,GER,1001285,=,38,(29),:,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61 202762 JOCHER Simon GER 1001285 = 38 (29) :...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388,20,USA,BAREFOOT,Daniel,4.72(=18),46.23,(16),94.14,(18),113.92,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20 USA BAREFOOT Daniel 4.72(=18) 46.23 (16) 94...
389,21,AUS,TIMMINGS,Nicholas,4.75,(23),46.15,(17),91.83,(24),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21 AUS TIMMINGS Nicholas 4.75 (23) 46.15 (17) ...
390,22,ISR,FIRESTONE,Jared,4.74,(22),45.78,(24),93.89,(19),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22 ISR FIRESTONE Jared 4.74 (22) 45.78 (24) 93...
391,23,JPN,TAKAHASHI,Hiroatsu,4.70,(16),46.11,(18),93.60,(22),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23 JPN TAKAHASHI Hiroatsu 4.70 (16) 46.11 (18)...
